# Generation of example data

The files in `examples/data` are based on data sets from [Liander open data](https://www.liander.nl/over-ons/open-data), 
specifically "Verbruiksdata slimme meter 2012-2014". This notebook downloads and prepares the data for use with
`PVDisaggregator`. The resulting files are part of the repository, you don't have to run this notebook
to run the example.

In [8]:
import os
from pathlib import Path

import hvplot.pandas  # noqa: F401
import pandas as pd

from sundael.disaggregation import PVDisaggregator

data_dir = Path("./data")
data_dir.mkdir(exist_ok=True)

In [9]:
zipfiles = [
    "over-liander-slimme-meter-dataset-2013-levering.zip",
    "over-liander-slimme-meter-dataset-2013-teruglevering.zip",
]

net_gen_file = data_dir / "Liander_2013_Zonnedael_slimme_meter_dataset_2013_Teruglevering.csv"
net_con_file = data_dir / "Liander_2013_Zonnedael_slimme_meter_dataset_2013_Levering.csv"

In [10]:
for zipfile in zipfiles:
    if not Path(zipfile).exists():
        !wget https://www.liander.nl/-/media/files/open-data/slimme-meter/{zipfile} --no-check-certificate

for _zipfile in zipfiles:
    !unzip {_zipfile}
    !rm {_zipfile}

os.rename("Zonnedael - slimme meter dataset - 2013 - Levering.csv", net_con_file)
os.rename("Zonnedael - slimme meter dataset - 2013 - Teruglevering.csv", net_gen_file)

--2026-06-30 08:59:26--  https://www.liander.nl/-/media/files/open-data/slimme-meter/over-liander-slimme-meter-dataset-2013-levering.zip
Resolving www.liander.nl (www.liander.nl)... 150.171.109.34, 2603:1061:14:20::1
Connecting to www.liander.nl (www.liander.nl)|150.171.109.34|:443... connected.
HTTP request sent, awaiting response... 200 OK
Cookie coming from www.liander.nl attempted to set domain to all-dxp-prd-cd.azurewebsites.net
Cookie coming from www.liander.nl attempted to set domain to all-dxp-prd-cd.azurewebsites.net
Length: 3910045 (3.7M) [application/x-zip-compressed]
Saving to: ‘over-liander-slimme-meter-dataset-2013-levering.zip’

over-liander-slimme 100%[===================>]   3.73M  5.99MB/s    in 0.6s    

2026-06-30 08:59:27 (5.99 MB/s) - ‘over-liander-slimme-meter-dataset-2013-levering.zip’ saved [3910045/3910045]

--2026-06-30 08:59:27--  https://www.liander.nl/-/media/files/open-data/slimme-meter/over-liander-slimme-meter-dataset-2013-teruglevering.zip
Resolving ww

# Create PV reference ratio

Ideally, you would retrieve reference PV data on a granular level, for instance per province or postal code area. 
You can use, for instance, data from [PVOutput](https://pvoutput.org/). Here, we use the net generation (export) data 
of all customersin this data set. We first calculate the ratio for all customers based on the net consumption profile. 
We then take the 95th percentile of all customers as the general ratio. This assumes that all customers are geographically
close, which would mean that we can assume that the maximum ratio at each timepoint is the "real" ratio and not influenced
by local weather conditions or consumption. This assumption is not always correct, but the resulting ratio will be closer
to the true ratio than assuming maximum solar intensity at each time point.

In [11]:
!ls data/

Liander_2013_Zonnedael_slimme_meter_dataset_2013_Levering.csv
Liander_2013_Zonnedael_slimme_meter_dataset_2013_Teruglevering.csv
net_con.parquet
net_gen.parquet
pv_ratio.parquet


In [12]:
# Read the generation input file
pv_ref = pd.read_csv(net_gen_file, sep=";", index_col=0, skiprows=1)
# Remove non-customer columns and customers with invalid profiles
pv_ref = pv_ref[[c for c in pv_ref.columns if c not in ["SOM", "terleveraars", "Klant 60", "Klant 18", "Klant 53"]]]
# Remove all customers that don't have generation
pv_ref = pv_ref.loc[:, pv_ref.max() > 1]
# Normalize
pv_ref = pv_ref / pv_ref.max()

# Convert index to DataTime format
pv_ref.index = pd.to_datetime(pv_ref.index, format="mixed", dayfirst=True).tz_localize(
    "Europe/Amsterdam", nonexistent="NaT", ambiguous="NaT"
)

# Only use the complete year 2013
pv_ref = pv_ref[(pv_ref.index.year == 2013) & (pv_ref.index != pd.NaT)]

# Drop duplicate time stamps
pv_ref = pv_ref.reset_index().drop_duplicates().set_index("datetime")

# Convert all profiles to ratio's
pv_ratio = PVDisaggregator.convert_pv_reference_to_ratio(pv_ref.T).T

# Take the 95th percentile of the ratio as reference and save the output
pv_ratio.quantile(0.95, axis=1).to_frame("average").to_parquet(data_dir / "pv_ratio.parquet")

2026-06-30 08:59:30 [info     ] Calculating pv ratio from pv reference
2026-06-30 08:59:30 [info     ] Setting temperature based on data from the Dutch meteorological institute.
2026-06-30 08:59:30 [info     ] This is not valid for other localities! Set auto_infer_temp to `False` 
2026-06-30 08:59:30 [info     ] to disable or provide temperature to PVDisaggregator.
2026-06-30 08:59:30 [info     ] Optimizing parameters         
2026-06-30 08:59:30 [info     ]   - Create parameter array    
2026-06-30 08:59:30 [info     ]   - Optimization              
2026-06-30 08:59:34 [info     ] Estimating generation...      
2026-06-30 08:59:34 [info     ]   - Area-dependent parameters 
2026-06-30 08:59:36 [info     ]   - Customer-dependent generation
2026-06-30 08:59:36 [info     ]   - Final calculation         
2026-06-30 08:59:36 [info     ] Done estimating generation    
2026-06-30 08:59:36 [info     ] Calculation generation and consumption
2026-06-30 08:59:36 [info     ] Done with disaggregati

In [13]:
# Example plot of the ratio's for all customers
pv_ratio.hvplot(width=1000, height=400)

:NdOverlay   [Variable]
   :Curve   [datetime]   (value)

# Create example data files

Create net consumption and net generation data files for all customers from the example data that have PV.

In [14]:
net_gen = pd.read_csv(net_gen_file, sep=";", index_col=0, skiprows=1)
net_con = pd.read_csv(net_con_file, sep=";", index_col=0)

idx = net_gen.loc[:, (net_gen.sum() > 100) & net_gen.columns.str.contains("Klant")].columns.intersection(
    net_con.columns
)

# Convert index to DataTime format
net_gen.index = pd.to_datetime(net_gen.index, format="mixed", dayfirst=True).tz_localize(
    "Europe/Amsterdam", nonexistent="NaT", ambiguous="NaT"
)
net_con.index = pd.to_datetime(net_con.index, format="mixed", dayfirst=True).tz_localize(
    "Europe/Amsterdam", nonexistent="NaT", ambiguous="NaT"
)

# Only use the complete year 2013
net_gen = net_gen.loc[(net_gen.index.year == 2013) & (net_gen.index != pd.NaT), idx]
net_con = net_con.loc[(net_con.index.year == 2013) & (net_con.index != pd.NaT), idx]

# Save files
net_gen.to_parquet(data_dir / "net_gen.parquet")
net_con.to_parquet(data_dir / "net_con.parquet")

/tmp/ipykernel_56254/1065469176.py:2: DtypeWarning: Columns (0,79,80) have mixed types. Specify dtype option on import or set low_memory=False.
  net_con = pd.read_csv(net_con_file, sep=";", index_col=0)


In [15]:
net_con.head()

,Klant 1,Klant 21,Klant 30,Klant 47,Klant 48,Klant 55,Klant 63,Klant 69,Klant 79,Klant 80
datetime,,,,,,,,,,
2013-01-01 00:00:00+01:00,71.0,24.0,36.0,107.0,128.0,144.0,109.0,107.0,223.0,203.0
2013-01-01 00:15:00+01:00,58.0,21.0,80.0,123.0,137.0,138.0,75.0,98.0,257.0,191.0
2013-01-01 00:30:00+01:00,57.0,13.0,41.0,119.0,98.0,149.0,77.0,91.0,179.0,162.0
2013-01-01 00:45:00+01:00,78.0,13.0,54.0,166.0,23.0,142.0,66.0,88.0,216.0,222.0
2013-01-01 01:00:00+01:00,63.0,41.0,56.0,102.0,28.0,168.0,85.0,89.0,209.0,211.0


In [16]:
net_gen.head()

,Klant 1,Klant 21,Klant 30,Klant 47,Klant 48,Klant 55,Klant 63,Klant 69,Klant 79,Klant 80
datetime,,,,,,,,,,
2013-01-01 00:00:00+01:00,0,0,0,0,0,0,0,0,0,0
2013-01-01 00:15:00+01:00,0,0,0,0,0,0,0,0,0,0
2013-01-01 00:30:00+01:00,0,0,0,0,0,0,0,0,0,0
2013-01-01 00:45:00+01:00,0,0,0,0,0,0,0,0,0,0
2013-01-01 01:00:00+01:00,0,0,0,0,0,0,0,0,0,0
